# Generate & Inspect Supporting Data

Explores the local corpora that drive the deterministic retrieval layer:
dependencies, incidents, ownership, runbooks, and the risk policy.

In [ ]:
from pathlib import Path
import json

data_dir = Path('..') / 'data'

## 1 — Corpus File Inventory

List every file in the data directory tree.

In [ ]:
for p in sorted(data_dir.rglob('*')):
    if p.is_file():
        rel = p.relative_to(data_dir)
        size_kb = p.stat().st_size / 1024
        print(f'  {str(rel):<50} {size_kb:>6.1f} KB')

## 2 — Dependencies Corpus

Show all dependency health signals indexed by service and environment.

In [ ]:
deps = json.loads((data_dir / 'dependencies' / 'dependencies.json').read_text())
print(f'Total dependency records: {len(deps)}\n')
for d in deps:
    print(f"  {d.get('name','?'):<25} service={d.get('service','?'):<18} "
          f"env={d.get('environment','?'):<12} status={d.get('status','?')}")

## 3 — Incidents Corpus

List recent incidents with severity, status, and affected service.

In [ ]:
incidents = json.loads((data_dir / 'incidents' / 'incidents.json').read_text())
print(f'Total incident records: {len(incidents)}\n')
for inc in incidents:
    print(f"  [{inc.get('severity','?')}] {inc.get('title','?'):<40} "
          f"status={inc.get('status','?'):<12} service={inc.get('service','?')}")

## 4 — Ownership Corpus

Show team ownership and on-call status per service.

In [ ]:
ownership = json.loads((data_dir / 'ownership' / 'ownership.json').read_text())
print(f'Total ownership records: {len(ownership)}\n')
for own in ownership:
    print(f"  {own.get('service','?'):<25} team={own.get('owning_team','?'):<20} "
          f"oncall={own.get('oncall_defined', '?')}")

## 5 — Runbooks

List available markdown runbook files and preview the first few lines of each.

In [ ]:
runbook_dir = data_dir / 'runbooks'
runbooks = sorted(runbook_dir.glob('*.md'))
print(f'Available runbooks: {len(runbooks)}\n')
for rb in runbooks:
    lines = rb.read_text().strip().splitlines()
    preview = lines[0] if lines else '(empty)'
    print(f'  {rb.name:<35} {preview}')

## 6 — Risk Policy Configuration

Display the current policy thresholds and coverage categories.

In [ ]:
import yaml

policy = yaml.safe_load((data_dir / 'policies' / 'risk_policy.yaml').read_text())
print('Policy version:', policy.get('policy_version', 'unknown'))
print('\nThresholds:')
for k, v in policy.get('thresholds', {}).items():
    print(f'  {k:<30} {v}')
print('\nCoverage categories:')
for cat in policy.get('coverage_categories', []):
    print(f'  - {cat}')

## 7 — Sample Bundles

List available sample bundles and show key fields from each.

In [ ]:
bundle_dir = data_dir / 'sample_bundles'
for bf in sorted(bundle_dir.glob('*.json')):
    bundle = json.loads(bf.read_text())
    print(f'\n=== {bf.stem} ===')
    for key in ['release_id', 'service', 'environment', 'ci_status',
                'change_freeze_active', 'rollback_plan_present', 'diff_size', 'approvals']:
        print(f'  {key:<25} {bundle.get(key, "N/A")}')